##Grant catalog permissions

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG bfsi_mdm_dev
TO `mdm_engineers`;

GRANT USE CATALOG
ON CATALOG bfsi_mdm_dev
TO `mdm_stewards`;

GRANT USE CATALOG
ON CATALOG bfsi_mdm_dev
TO `mdm_analysts`;

##Grant schema permissions

###Engineers

In [0]:
%sql

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.bronze
TO `mdm_engineers`;

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.silver
TO `mdm_engineers`;

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.mdm
TO `mdm_engineers`;

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.gold
TO `mdm_engineers`;

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.audit
TO `mdm_engineers`;

###Analysts

In [0]:
%sql

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.gold
TO `mdm_analysts`;

###Stewards

In [0]:
%sql

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.silver
TO `mdm_stewards`;

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.mdm
TO `mdm_stewards`;

GRANT USE SCHEMA
ON SCHEMA bfsi_mdm_dev.gold
TO `mdm_stewards`;

##Grant table-level access

###Analysts

Analysts should read the Golden Record but not modify it.

In [0]:
%sql

GRANT SELECT
ON TABLE bfsi_mdm_dev.gold.master_client
TO `mdm_analysts`;

GRANT SELECT
ON TABLE bfsi_mdm_dev.gold.client_source_map
TO `mdm_analysts`;

###Stewards



In [0]:
%sql

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.silver
TO `mdm_stewards`;

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.mdm
TO `mdm_stewards`;

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.gold
TO `mdm_stewards`;


###Engineers

In [0]:
%sql

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.bronze
TO `mdm_engineers`;

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.silver
TO `mdm_engineers`;

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.mdm
TO `mdm_engineers`;

GRANT SELECT
ON SCHEMA bfsi_mdm_dev.gold
TO `mdm_engineers`;

##Verify RBAC

In [0]:
%sql
SHOW GRANTS ON CATALOG bfsi_mdm_dev;

Principal,ActionType,ObjectType,ObjectKey
0827b320-d844-4426-81ba-be00cafc3a05,SELECT,CATALOG,bfsi_mdm_dev
0827b320-d844-4426-81ba-be00cafc3a05,USE CATALOG,CATALOG,bfsi_mdm_dev
0827b320-d844-4426-81ba-be00cafc3a05,USE SCHEMA,CATALOG,bfsi_mdm_dev
account users,BROWSE,CATALOG,bfsi_mdm_dev
mdm_engineers,USE CATALOG,CATALOG,bfsi_mdm_dev
mdm_analysts,USE CATALOG,CATALOG,bfsi_mdm_dev
mdm_stewards,USE CATALOG,CATALOG,bfsi_mdm_dev


In [0]:
%sql
SHOW GRANTS ON SCHEMA bfsi_mdm_dev.gold;

Principal,ActionType,ObjectType,ObjectKey
0827b320-d844-4426-81ba-be00cafc3a05,SELECT,CATALOG,bfsi_mdm_dev
0827b320-d844-4426-81ba-be00cafc3a05,USE SCHEMA,CATALOG,bfsi_mdm_dev
mdm_engineers,SELECT,SCHEMA,bfsi_mdm_dev.gold
mdm_engineers,USE SCHEMA,SCHEMA,bfsi_mdm_dev.gold
mdm_analysts,USE SCHEMA,SCHEMA,bfsi_mdm_dev.gold
mdm_stewards,SELECT,SCHEMA,bfsi_mdm_dev.gold
mdm_stewards,USE SCHEMA,SCHEMA,bfsi_mdm_dev.gold


In [0]:
%sql
SHOW GRANTS ON TABLE bfsi_mdm_dev.gold.master_client;

Principal,ActionType,ObjectType,ObjectKey
0827b320-d844-4426-81ba-be00cafc3a05,SELECT,CATALOG,bfsi_mdm_dev
mdm_engineers,SELECT,SCHEMA,bfsi_mdm_dev.gold
mdm_analysts,SELECT,TABLE,bfsi_mdm_dev.gold.master_client
mdm_stewards,SELECT,SCHEMA,bfsi_mdm_dev.gold


##Define the PII classification

Now we classify sensitive Golden Record attributes.\
The planned taxonomy is:

| Column    | Classification            |
| --------- | ------------------------- |
| `name`    | `pii = direct_identifier` |
| `phone`   | `pii = phone`             |
| `email`   | `pii = email`             |
| `address` | `pii = address`           |
| `tax_id`  | `pii = tax_id`            |
| `aum`     | `sensitivity = financial` |



##Apply PII tags

In [0]:
%sql

ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN name
SET TAGS ('pii' = 'direct_identifier');

ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN phone
SET TAGS ('pii' = 'phone');

ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN email
SET TAGS ('pii' = 'email');

ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN address
SET TAGS ('pii' = 'address');

ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN tax_id
SET TAGS ('pii' = 'tax_id');

ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN aum
SET TAGS ('sensitivity' = 'financial');

##Verify PII tags

In [0]:
%sql

SELECT
    catalog_name,
    schema_name,
    table_name,
    column_name,
    tag_name,
    tag_value
FROM bfsi_mdm_dev.information_schema.column_tags
WHERE schema_name = 'gold'
  AND table_name = 'master_client'
ORDER BY column_name;

catalog_name,schema_name,table_name,column_name,tag_name,tag_value
bfsi_mdm_dev,gold,master_client,address,pii,address
bfsi_mdm_dev,gold,master_client,aum,sensitivity,financial
bfsi_mdm_dev,gold,master_client,email,pii,email
bfsi_mdm_dev,gold,master_client,name,pii,direct_identifier
bfsi_mdm_dev,gold,master_client,phone,pii,phone
bfsi_mdm_dev,gold,master_client,tax_id,pii,tax_id


##Create the email masking function

In [0]:
%sql

CREATE OR REPLACE FUNCTION bfsi_mdm_dev.gold.mask_email(value STRING)
RETURNS STRING
RETURN
    CASE
        WHEN is_account_group_member('mdm_pii_access')
            THEN value

        WHEN value IS NULL
            THEN NULL

        WHEN instr(value, '@') > 1
            THEN concat(
                left(value, 1),
                '***@',
                substring(
                    value,
                    instr(value, '@') + 1,
                    length(value)
                )
            )

        ELSE
            '***'
    END;


##Create the phone masking function

In [0]:
%sql

CREATE OR REPLACE FUNCTION bfsi_mdm_dev.gold.mask_phone(value STRING)
RETURNS STRING
RETURN
    CASE
        WHEN is_account_group_member('mdm_pii_access')
            THEN value

        WHEN value IS NULL
            THEN NULL

        WHEN length(value) >= 4
            THEN concat(
                '******',
                right(value, 4)
            )

        ELSE
            '******'
    END;

##Create the Tax ID masking function

In [0]:
%sql
CREATE OR REPLACE FUNCTION bfsi_mdm_dev.gold.mask_tax_id(value STRING)
RETURNS STRING
RETURN
    CASE
        WHEN is_account_group_member('mdm_pii_access')
            THEN value

        WHEN value IS NULL
            THEN NULL

        WHEN length(value) >= 4
            THEN concat(
                '********',
                right(value, 4)
            )

        ELSE
            '********'
    END;

##Apply the masks

In [0]:
%sql
ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN email
SET MASK bfsi_mdm_dev.gold.mask_email;

In [0]:
%sql
ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN phone
SET MASK bfsi_mdm_dev.gold.mask_phone;

In [0]:
%sql
ALTER TABLE bfsi_mdm_dev.gold.master_client
ALTER COLUMN tax_id
SET MASK bfsi_mdm_dev.gold.mask_tax_id;

##Test masking

###Test A — Unauthorized user

In [0]:
%sql
SELECT
    master_client_id,
    name,
    phone,
    email,
    tax_id
FROM bfsi_mdm_dev.gold.master_client
LIMIT 10;

master_client_id,name,phone,email,tax_id
MC000003,JEMMA DEAN,******0356,l***@example.net,********003A
MC000007,MITCHELL JANICE BLACKBURN,******0059,d***@example.com,********007A
MC000009,Leanne Robert Stevens,******0874,r***@example.com,null
MC000012,Molly Webb,******0814,h***@example.com,null
MC000016,LEIGH BURTON,******8951,v***@example.net,********018A
MC000017,William Francesca Allan,******0479,b***@example.org,null
MC000018,GEOFFREY WEST,******0710,u***@example.net,********020A
MC000021,JOAN HUNT,******0116,l***@example.net,********023A
MC000023,LEONARD THOMAS SHAH,******0218,m***@example.com,********025A
MC000026,Joe Saunders,******8419,r***@example.com,null


###Test B — Authorized PII user

In [0]:
%sql
SELECT
    master_client_id,
    name,
    phone,
    email,
    tax_id
FROM bfsi_mdm_dev.gold.master_client
LIMIT 10;

master_client_id,name,phone,email,tax_id
MC000003,JEMMA DEAN,+44117 4960356,lewisbrian@example.net,QQ0000003A
MC000007,MITCHELL JANICE BLACKBURN,(0113)4960059,darrendouglas@example.com,QQ0000007A
MC000009,Leanne Robert Stevens,+44(0)9098790874,robertscarol@example.com,null
MC000012,Molly Webb,+44(0)116 496 0814,howard15@example.com,null
MC000016,LEIGH BURTON,029 2018951,vmurray@example.net,QQ0000018A
MC000017,William Francesca Allan,+44(0)118 496 0479,brookskatie@example.org,null
MC000018,GEOFFREY WEST,+44(0)2074960710,uwhite@example.net,QQ0000020A
MC000021,JOAN HUNT,(020) 7496 0116,liam07@example.net,QQ0000023A
MC000023,LEONARD THOMAS SHAH,0115 496 0218,mauricesmith@example.com,QQ0000025A
MC000026,Joe Saunders,(029)2018419,racheljones@example.com,null


##Audit verification

In [0]:
%sql

SELECT
    run_id,
    published_at,
    status,
    history_rows,
    validation_status
FROM bfsi_mdm_dev.audit.pipeline_run_audit
ORDER BY published_at DESC
LIMIT 20;

run_id,published_at,status,history_rows,validation_status
efdcf409-3420-41df-b97e-2b40f684e1f5,2026-09-28T22:26:26.661785Z,SUCCESS,14047696,ALL_GOLD_VALIDATIONS_PASSED
ea0c6e1c-0b53-4464-ac49-e27105634755,2026-09-28T18:54:07.002002Z,SUCCESS,14047696,ALL_GOLD_VALIDATIONS_PASSED


In [0]:
%sql

SELECT
    COUNT(*) AS total_audit_runs,
    SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_runs,
    SUM(CASE WHEN validation_status = 'PASS' THEN 1 ELSE 0 END) AS passed_validations
FROM bfsi_mdm_dev.audit.pipeline_run_audit;

total_audit_runs,successful_runs,passed_validations
2,2,0


In [0]:
%sql

SELECT
    COUNT(*) AS total_audit_runs,

    SUM(
        CASE
            WHEN status = 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS successful_runs,

    SUM(
        CASE
            WHEN validation_status = 'ALL_GOLD_VALIDATIONS_PASSED'
            THEN 1
            ELSE 0
        END
    ) AS passed_validations

FROM bfsi_mdm_dev.audit.pipeline_run_audit;

total_audit_runs,successful_runs,passed_validations
2,2,2
